In [28]:
import pandas as pd
from datasets import Dataset

from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

import ast

In [2]:
path = '/Users/jangsehyeon/Downloads/sft_dataset.csv'

df = pd.read_csv(path)

In [3]:
### 확인용 코드
def validate_messages(messages):
    """메시지 형식 검증"""
    if not isinstance(messages, list):
        return False
    
    for msg in messages:
        if not isinstance(msg, dict):
            return False
        if 'role' not in msg or 'content' not in msg:
            return False
        if msg['role'] not in ['system', 'user', 'assistant']:
            return False
    
    return True

valid_mask = df['messages'].apply(validate_messages)
print(f"유효한 샘플: {valid_mask.sum()}/{len(df)}")

유효한 샘플: 0/38334


In [29]:
def parse_messages(ex):
    # 문자열 -> list[dict]
    m = ex["messages"]
    if isinstance(m, str):
        ex["messages"] = ast.literal_eval(m) 
    return ex


In [30]:
### 데이터셋 분리
dataset = Dataset.from_pandas(df[['messages']], preserve_index=False)
dataset = dataset.map(parse_messages)
dataset = dataset.train_test_split(test_size=0.1, seed=42)

print(f"Train 샘플: {len(dataset['train'])}")
print(f"Val 샘플: {len(dataset['test'])}")

Map:   0%|          | 0/38334 [00:00<?, ? examples/s]

Train 샘플: 34500
Val 샘플: 3834


In [68]:
dataset['train'][0]

{'messages': [{'content': '어제 남친이랑 제10회 춘천영화제 다녀왔잖아.', 'role': 'user'},
  {'content': '짜짜, 나도 가고 싶었는데 어땠어?', 'role': 'assistant'},
  {'content': '어제 개막식이라 사람들 엄청 많드라.', 'role': 'user'},
  {'content': '어제가 개막식 이었어? 아까비 나도 개막식 가고 싶었는데, 이번 영화제에는 어떤 연예인이 참석해?',
   'role': 'assistant'},
  {'content': '옥자연이 사회를 맡고 이준익 감독, 안성기, 박중훈, 이준기 배우가 관객과의 대화에 참여 한대.',
   'role': 'user'},
  {'content': '대박, 나 이준기 엄청 좋아하잖아.', 'role': 'assistant'},
  {'content': 'ㅋㅋ 너 생각 나더라, 이준기 연기 정말 잘하지.', 'role': 'user'},
  {'content': '나도 이번엔 무적권 가볼 거야, 언제가 폐막식이지?', 'role': 'assistant'},
  {'content': '9월 11일 월요일까지 아직 시간 있으니까 꼭 가봐.', 'role': 'user'},
  {'content': '영화는 어디에서 볼 수 있는 거야?', 'role': 'assistant'},
  {'content': '공지천 아울러 까페와 메가박스 남춘천점, 커먼즈필드 안녕하우스에서 상영돼.', 'role': 'user'},
  {'content': '이번 영화제 주제는 뭐야?', 'role': 'assistant'},
  {'content': '영화의 봄, 낭만의 가을이 이번 영화제 주제야.', 'role': 'user'},
  {'content': '내가 좋아하는 로맨스 드라마 많았으면 좋겠다.', 'role': 'assistant'},
  {'content': '응, 요일별 상영 영화 보내줄 테니까 보고 가봐.', 'role': 'user

In [31]:
model_name = "Qwen/Qwen3-4B-Instruct-2507"

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    trust_remote_code=True
)

# Padding 설정
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

In [32]:
dataset['train'][0]['messages']

[{'content': '어제 남친이랑 제10회 춘천영화제 다녀왔잖아.', 'role': 'user'},
 {'content': '짜짜, 나도 가고 싶었는데 어땠어?', 'role': 'assistant'},
 {'content': '어제 개막식이라 사람들 엄청 많드라.', 'role': 'user'},
 {'content': '어제가 개막식 이었어? 아까비 나도 개막식 가고 싶었는데, 이번 영화제에는 어떤 연예인이 참석해?',
  'role': 'assistant'},
 {'content': '옥자연이 사회를 맡고 이준익 감독, 안성기, 박중훈, 이준기 배우가 관객과의 대화에 참여 한대.',
  'role': 'user'},
 {'content': '대박, 나 이준기 엄청 좋아하잖아.', 'role': 'assistant'},
 {'content': 'ㅋㅋ 너 생각 나더라, 이준기 연기 정말 잘하지.', 'role': 'user'},
 {'content': '나도 이번엔 무적권 가볼 거야, 언제가 폐막식이지?', 'role': 'assistant'},
 {'content': '9월 11일 월요일까지 아직 시간 있으니까 꼭 가봐.', 'role': 'user'},
 {'content': '영화는 어디에서 볼 수 있는 거야?', 'role': 'assistant'},
 {'content': '공지천 아울러 까페와 메가박스 남춘천점, 커먼즈필드 안녕하우스에서 상영돼.', 'role': 'user'},
 {'content': '이번 영화제 주제는 뭐야?', 'role': 'assistant'},
 {'content': '영화의 봄, 낭만의 가을이 이번 영화제 주제야.', 'role': 'user'},
 {'content': '내가 좋아하는 로맨스 드라마 많았으면 좋겠다.', 'role': 'assistant'},
 {'content': '응, 요일별 상영 영화 보내줄 테니까 보고 가봐.', 'role': 'user'},
 {'content': '레알 고마워 칭구, 

In [35]:
# Chat Template 확인
print("Chat template 테스트:")
test_messages = dataset["train"][0]["messages"]
print(type(test_messages))
formatted = tokenizer.apply_chat_template(
    test_messages,
    tokenize=False,
    add_generation_prompt=False
)

print(formatted)
print("\n" + "="*50 + "\n")

Chat template 테스트:
<class 'list'>
<|im_start|>user
어제 남친이랑 제10회 춘천영화제 다녀왔잖아.<|im_end|>
<|im_start|>assistant
짜짜, 나도 가고 싶었는데 어땠어?<|im_end|>
<|im_start|>user
어제 개막식이라 사람들 엄청 많드라.<|im_end|>
<|im_start|>assistant
어제가 개막식 이었어? 아까비 나도 개막식 가고 싶었는데, 이번 영화제에는 어떤 연예인이 참석해?<|im_end|>
<|im_start|>user
옥자연이 사회를 맡고 이준익 감독, 안성기, 박중훈, 이준기 배우가 관객과의 대화에 참여 한대.<|im_end|>
<|im_start|>assistant
대박, 나 이준기 엄청 좋아하잖아.<|im_end|>
<|im_start|>user
ㅋㅋ 너 생각 나더라, 이준기 연기 정말 잘하지.<|im_end|>
<|im_start|>assistant
나도 이번엔 무적권 가볼 거야, 언제가 폐막식이지?<|im_end|>
<|im_start|>user
9월 11일 월요일까지 아직 시간 있으니까 꼭 가봐.<|im_end|>
<|im_start|>assistant
영화는 어디에서 볼 수 있는 거야?<|im_end|>
<|im_start|>user
공지천 아울러 까페와 메가박스 남춘천점, 커먼즈필드 안녕하우스에서 상영돼.<|im_end|>
<|im_start|>assistant
이번 영화제 주제는 뭐야?<|im_end|>
<|im_start|>user
영화의 봄, 낭만의 가을이 이번 영화제 주제야.<|im_end|>
<|im_start|>assistant
내가 좋아하는 로맨스 드라마 많았으면 좋겠다.<|im_end|>
<|im_start|>user
응, 요일별 상영 영화 보내줄 테니까 보고 가봐.<|im_end|>
<|im_start|>assistant
레알 고마워 칭구, 이번 영화제 가서 가을낭만 제대로 느끼고 와야지 ㅋㅋ<|im_end|>





In [ ]:

# 3. 모델 로드
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,  # 메모리 절약
    device_map="auto",  # 자동으로 GPU 할당
    trust_remote_code=True,
    attn_implementation="flash_attention_2",  # 더 빠름 (선택)
)

print(f"모델 파라미터: {model.num_parameters() / 1e9:.2f}B")
print(f"모델 dtype: {model.dtype}")

In [37]:
test_messages = dataset["train"][0]["messages"]
print("Messages 개수:", len(test_messages))
print("\n각 메시지:")
for i, msg in enumerate(test_messages):
    print(f"[{i}] {msg['role']}: {msg['content'][:50]}...")

formatted = tokenizer.apply_chat_template(
    test_messages,
    tokenize=False,
    add_generation_prompt=False
)
print("\n=== Formatted Text ===")
print(formatted)

encoded = tokenizer.apply_chat_template(
    test_messages,
    tokenize=True,
    add_generation_prompt=False,
    return_dict=True,
    return_tensors="pt"
)
print("\n=== Tokenized ===")
print("Input IDs shape:", encoded['input_ids'].shape)
print("Attention mask shape:", encoded['attention_mask'].shape)

Messages 개수: 16

각 메시지:
[0] user: 어제 남친이랑 제10회 춘천영화제 다녀왔잖아....
[1] assistant: 짜짜, 나도 가고 싶었는데 어땠어?...
[2] user: 어제 개막식이라 사람들 엄청 많드라....
[3] assistant: 어제가 개막식 이었어? 아까비 나도 개막식 가고 싶었는데, 이번 영화제에는 어떤 연예인이 ...
[4] user: 옥자연이 사회를 맡고 이준익 감독, 안성기, 박중훈, 이준기 배우가 관객과의 대화에 참여 ...
[5] assistant: 대박, 나 이준기 엄청 좋아하잖아....
[6] user: ㅋㅋ 너 생각 나더라, 이준기 연기 정말 잘하지....
[7] assistant: 나도 이번엔 무적권 가볼 거야, 언제가 폐막식이지?...
[8] user: 9월 11일 월요일까지 아직 시간 있으니까 꼭 가봐....
[9] assistant: 영화는 어디에서 볼 수 있는 거야?...
[10] user: 공지천 아울러 까페와 메가박스 남춘천점, 커먼즈필드 안녕하우스에서 상영돼....
[11] assistant: 이번 영화제 주제는 뭐야?...
[12] user: 영화의 봄, 낭만의 가을이 이번 영화제 주제야....
[13] assistant: 내가 좋아하는 로맨스 드라마 많았으면 좋겠다....
[14] user: 응, 요일별 상영 영화 보내줄 테니까 보고 가봐....
[15] assistant: 레알 고마워 칭구, 이번 영화제 가서 가을낭만 제대로 느끼고 와야지 ㅋㅋ...

=== Formatted Text ===
<|im_start|>user
어제 남친이랑 제10회 춘천영화제 다녀왔잖아.<|im_end|>
<|im_start|>assistant
짜짜, 나도 가고 싶었는데 어땠어?<|im_end|>
<|im_start|>user
어제 개막식이라 사람들 엄청 많드라.<|im_end|>
<|im_start|>assistant
어제가 개막식 이었어? 아까비 나도 개막식 가고 싶었는데, 이번 영화제에는 어떤 연예인

In [42]:
from trl import DataCollatorForCompletionOnlyLM
from trl import SFTTrainer, SFTConfig

# Qwen: "<|im_start|>assistant\n"
# Llama: "<|start_header_id|>assistant<|end_header_id|>\n\n"

response_template = "<|im_start|>assistant\n"  # Qwen 기준
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
)

collator = DataCollatorForCompletionOnlyLM(
    response_template=response_template,
    tokenizer=tokenizer,
    mlm=False
)


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the disk.


In [ ]:
instruction_template = "<|im_start|>user"
response_template = "<|im_start|>assistant"  # Qwen 기준

collator = DataCollatorForCompletionOnlyLM(
    instruction_template=instruction_template,
    response_template=response_template,
    tokenizer=tokenizer,
)


In [44]:
# 테스트
test_batch = [
    tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )
]

# Collator 적용
from transformers import DefaultDataCollator
inputs = tokenizer(test_batch, return_tensors="pt", padding=True)

# Label 확인
features = [{"input_ids": inputs["input_ids"][0]}]
batch = collator(features)

print("Input IDs:", batch['input_ids'].shape)
print("Labels:", batch['labels'].shape)

# -100인 위치 확인 (loss 무시되는 부분)
labels = batch['labels'][0]
print(f"\n전체 토큰: {len(labels)}")
print(f"학습 토큰 (-100 아닌 것): {(labels != -100).sum().item()}")
print(f"무시 토큰 (-100): {(labels == -100).sum().item()}")

Input IDs: torch.Size([1, 35])
Labels: torch.Size([1, 35])

전체 토큰: 35
학습 토큰 (-100 아닌 것): 15
무시 토큰 (-100): 20


In [ ]:
instruction_template = "<|im_start|>user"

# 'assistant' 시작 부분 (여기서부터 마스킹 해제 -> 학습)
response_template = "<|im_start|>assistant"

In [60]:
### 실험용
tokenizer = AutoTokenizer.from_pretrained(model_name)

# 멀티턴 메시지
test_messages = dataset["train"][0]["messages"]

# Chat template 적용
formatted_text = tokenizer.apply_chat_template(
    test_messages,
    tokenize=False,
    add_generation_prompt=False
)

print("=== Formatted Text ===")
print(formatted_text)
print("\n" + "="*80 + "\n")

response_template = "<|im_start|>assistant\n"

# collator = DataCollatorForCompletionOnlyLM(
#     response_template=response_template,
#     tokenizer=tokenizer,
# )

collator = DataCollatorForCompletionOnlyLM(
    instruction_template=instruction_template,
    response_template=response_template,
    tokenizer=tokenizer,
    mlm=False,
)

tokenized = tokenizer(
    formatted_text,
    return_tensors="pt",
    truncation=True,
    max_length=512,
)

features = [{"input_ids": tokenized["input_ids"][0]}]
batch = collator(features)

input_ids = batch['input_ids'][0]
labels = batch['labels'][0]

print(f"총 토큰 수: {len(input_ids)}")
print(f"학습 토큰 (label != -100): {(labels != -100).sum().item()}")
print(f"무시 토큰 (label == -100): {(labels == -100).sum().item()}")

# === 각 응답별로 확인 ===
print("\n" + "="*80)
print("각 턴별 학습 여부:")
print("="*80)

current_text = ""
learning_segments = []
non_learning_segments = []

for i, (token_id, label) in enumerate(zip(input_ids, labels)):
    token_text = tokenizer.decode([token_id])
    
    if label != -100:
        current_text += token_text
    else:
        if current_text:
            learning_segments.append(current_text)
            current_text = ""

if current_text:
    learning_segments.append(current_text)

print("\n학습되는 부분 (assistant 응답들):")
for idx, seg in enumerate(learning_segments, 1):
    print(f"\n[응답 {idx}]")
    print(seg.strip())

print("\n" + "="*80)
print("토큰 레벨 분석 (처음 200개 토큰만):")
print("="*80)

for i in range(min(200, len(input_ids))):
    token = tokenizer.decode([input_ids[i]])
    label_status = "학습" if labels[i] != -100 else "무시"
    print(f"[{i:3d}] {label_status} | {repr(token)}")


=== Formatted Text ===
<|im_start|>user
어제 남친이랑 제10회 춘천영화제 다녀왔잖아.<|im_end|>
<|im_start|>assistant
짜짜, 나도 가고 싶었는데 어땠어?<|im_end|>
<|im_start|>user
어제 개막식이라 사람들 엄청 많드라.<|im_end|>
<|im_start|>assistant
어제가 개막식 이었어? 아까비 나도 개막식 가고 싶었는데, 이번 영화제에는 어떤 연예인이 참석해?<|im_end|>
<|im_start|>user
옥자연이 사회를 맡고 이준익 감독, 안성기, 박중훈, 이준기 배우가 관객과의 대화에 참여 한대.<|im_end|>
<|im_start|>assistant
대박, 나 이준기 엄청 좋아하잖아.<|im_end|>
<|im_start|>user
ㅋㅋ 너 생각 나더라, 이준기 연기 정말 잘하지.<|im_end|>
<|im_start|>assistant
나도 이번엔 무적권 가볼 거야, 언제가 폐막식이지?<|im_end|>
<|im_start|>user
9월 11일 월요일까지 아직 시간 있으니까 꼭 가봐.<|im_end|>
<|im_start|>assistant
영화는 어디에서 볼 수 있는 거야?<|im_end|>
<|im_start|>user
공지천 아울러 까페와 메가박스 남춘천점, 커먼즈필드 안녕하우스에서 상영돼.<|im_end|>
<|im_start|>assistant
이번 영화제 주제는 뭐야?<|im_end|>
<|im_start|>user
영화의 봄, 낭만의 가을이 이번 영화제 주제야.<|im_end|>
<|im_start|>assistant
내가 좋아하는 로맨스 드라마 많았으면 좋겠다.<|im_end|>
<|im_start|>user
응, 요일별 상영 영화 보내줄 테니까 보고 가봐.<|im_end|>
<|im_start|>assistant
레알 고마워 칭구, 이번 영화제 가서 가을낭만 제대로 느끼고 와야지 ㅋㅋ<|im_end|>



총 토큰 수: 424
학습 토큰 (l

In [57]:
# Response template 자동 추출
def find_response_template(tokenizer, test_messages):
    """Chat template에서 response template 자동 추출"""
    
    # 짧은 대화로 테스트
    short_messages = [
        {"role": "user", "content": "안녕"},
        {"role": "assistant", "content": "응"}
    ]
    
    formatted = tokenizer.apply_chat_template(
        short_messages,
        tokenize=False,
        add_generation_prompt=False
    )
    
    print("=== Chat Template 분석 ===")
    print(formatted)
    print("="*50)
    
    # Assistant 시작 부분 찾기
    lines = formatted.split('\n')
    for i, line in enumerate(lines):
        if 'assistant' in line:
            # 다음 줄이 실제 응답
            response_start = '\n'.join(lines[:i+2])
            # "응" 바로 전까지가 template
            template_end = response_start.rfind('응')
            response_template = response_start[:template_end]
            
            # 마지막 줄바꿈까지 포함
            if not response_template.endswith('\n'):
                # 이미 있는 경우도 있음
                pass
            
            print(f"\n✅ Response template: {repr(response_template)}")
            return response_template
    
    return None

# 자동 탐지
response_template = find_response_template(tokenizer, test_messages)

# 또는 수동으로 확인
print("\n=== 수동 확인 ===")
short_test = [
    {"role": "user", "content": "테스트"},
    {"role": "assistant", "content": "응답"}
]
formatted = tokenizer.apply_chat_template(short_test, tokenize=False)
print(formatted)
print("\n위 결과를 보고 '<|im_start|>assistant' 이후 '응답' 전까지를 복사하세요!")

=== Chat Template 분석 ===
<|im_start|>user
안녕<|im_end|>
<|im_start|>assistant
응<|im_end|>


✅ Response template: '<|im_start|>user\n안녕<|im_end|>\n<|im_start|>assistant\n'

=== 수동 확인 ===
<|im_start|>user
테스트<|im_end|>
<|im_start|>assistant
응답<|im_end|>


위 결과를 보고 '<|im_start|>assistant' 이후 '응답' 전까지를 복사하세요!


In [58]:
response_template = "<|im_start|>assistant\n"

# 이게 정말 제대로 적용되었는지 확인
print(f"Response template: {repr(response_template)}")
print(f"Template 길이: {len(response_template)}")

Response template: '<|im_start|>assistant\n'
Template 길이: 22


In [59]:
from transformers import AutoTokenizer
from trl import DataCollatorForCompletionOnlyLM
import torch

tokenizer = AutoTokenizer.from_pretrained(model_name)

# 테스트 메시지
test_messages = dataset["train"][0]["messages"]

# Chat template 적용
formatted_text = tokenizer.apply_chat_template(
    test_messages,
    tokenize=False,
    add_generation_prompt=False
)

print("=== 1. Formatted Text 확인 ===")
print(formatted_text[:500])  # 처음 500자만
print("\n" + "="*80 + "\n")

# Response template 설정
response_template = "<|im_start|>assistant\n"
print(f"=== 2. Response Template ===")
print(f"Template: {repr(response_template)}")

# Formatted text에 response template이 몇 번 나오는지 확인
count = formatted_text.count(response_template)
print(f"Formatted text에서 발견된 횟수: {count}번")
print("\n" + "="*80 + "\n")

# 실제로 어디에 있는지 확인
import re
positions = [m.start() for m in re.finditer(re.escape(response_template), formatted_text)]
print(f"=== 3. Response Template 위치 ===")
print(f"발견된 위치들: {positions}")

if len(positions) > 0:
    # 첫 번째 response template 주변 확인
    pos = positions[0]
    print(f"\n첫 번째 발견 위치 ({pos}) 주변:")
    print(repr(formatted_text[max(0, pos-20):pos+50]))
print("\n" + "="*80 + "\n")

# === 핵심: Response template을 토크나이징했을 때 ===
response_template_ids = tokenizer.encode(response_template, add_special_tokens=False)
print(f"=== 4. Response Template 토크나이징 ===")
print(f"Response template IDs: {response_template_ids}")
print(f"토큰 개수: {len(response_template_ids)}")
print(f"디코딩: {repr(tokenizer.decode(response_template_ids))}")
print("\n" + "="*80 + "\n")

# Collator 생성
collator = DataCollatorForCompletionOnlyLM(
    response_template=response_template,
    tokenizer=tokenizer,
)

# === 5. DataCollator 내부 확인 ===
print("=== 5. DataCollator 설정 확인 ===")
print(f"Collator response_template: {repr(collator.response_template)}")
print(f"Collator response_token_ids: {collator.response_token_ids}")
print("\n" + "="*80 + "\n")

# 토크나이징
tokenized = tokenizer(
    formatted_text,
    return_tensors="pt",
    truncation=True,
    max_length=512,
)

print("=== 6. 전체 Text 토크나이징 ===")
print(f"Input IDs shape: {tokenized['input_ids'].shape}")
print(f"총 토큰 수: {len(tokenized['input_ids'][0])}")

# Collator 적용
features = [{"input_ids": tokenized["input_ids"][0]}]
batch = collator(features)

input_ids = batch['input_ids'][0]
labels = batch['labels'][0]

print(f"\n=== 7. Label 생성 결과 ===")
print(f"총 토큰 수: {len(input_ids)}")
print(f"학습 토큰 (label != -100): {(labels != -100).sum().item()}")
print(f"무시 토큰 (label == -100): {(labels == -100).sum().item()}")

# === 8. Response template ID가 실제 input_ids에 있는지 확인 ===
print(f"\n=== 8. Response Template ID 매칭 확인 ===")
response_token_ids = collator.response_token_ids

# input_ids를 list로 변환
input_ids_list = input_ids.tolist()

# Response template이 나타나는 모든 위치 찾기
def find_subsequence(seq, subseq):
    """리스트에서 부분 시퀀스의 모든 위치 찾기"""
    positions = []
    subseq_len = len(subseq)
    for i in range(len(seq) - subseq_len + 1):
        if seq[i:i+subseq_len] == subseq:
            positions.append(i)
    return positions

found_positions = find_subsequence(input_ids_list, response_token_ids)
print(f"Input IDs에서 response template 발견 횟수: {len(found_positions)}번")
print(f"발견된 위치들: {found_positions}")

# 각 위치 주변 확인
for idx, pos in enumerate(found_positions[:3]):  # 처음 3개만
    print(f"\n[발견 {idx+1}] 위치 {pos}:")
    # 위치 이후 학습되는지 확인
    after_template_labels = labels[pos:pos+20]
    print(f"  Template 이후 20개 토큰의 label: {after_template_labels.tolist()}")
    
    # 실제 텍스트
    after_ids = input_ids[pos:pos+20]
    print(f"  텍스트: {repr(tokenizer.decode(after_ids))}")

print("\n" + "="*80 + "\n")

# === 9. 학습되는 부분 디코딩 ===
print("=== 9. 실제 학습되는 텍스트 ===")
learning_mask = labels != -100
learning_ids = input_ids[learning_mask]
print(tokenizer.decode(learning_ids, skip_special_tokens=False))

=== 1. Formatted Text 확인 ===
<|im_start|>user
어제 남친이랑 제10회 춘천영화제 다녀왔잖아.<|im_end|>
<|im_start|>assistant
짜짜, 나도 가고 싶었는데 어땠어?<|im_end|>
<|im_start|>user
어제 개막식이라 사람들 엄청 많드라.<|im_end|>
<|im_start|>assistant
어제가 개막식 이었어? 아까비 나도 개막식 가고 싶었는데, 이번 영화제에는 어떤 연예인이 참석해?<|im_end|>
<|im_start|>user
옥자연이 사회를 맡고 이준익 감독, 안성기, 박중훈, 이준기 배우가 관객과의 대화에 참여 한대.<|im_end|>
<|im_start|>assistant
대박, 나 이준기 엄청 좋아하잖아.<|im_end|>
<|im_start|>user
ㅋㅋ 너 생각 나더라, 이준기 연기 정말 잘하지.<|im_end|>
<|im_start|>assistant
나도 이번엔 무적권 가볼 거야, 언제가 폐막식이지?<|im_end|>
<|im_start|


=== 2. Response Template ===
Template: '<|im_start|>assistant\n'
Formatted text에서 발견된 횟수: 8번


=== 3. Response Template 위치 ===
발견된 위치들: [53, 153, 321, 428, 546, 667, 768, 881]

첫 번째 발견 위치 (53) 주변:
'화제 다녀왔잖아.<|im_end|>\n<|im_start|>assistant\n짜짜, 나도 가고 싶었는데 어땠어?<|im_end|'


=== 4. Response Template 토크나이징 ===
Response template IDs: [151644, 77091, 198]
토큰 개수: 3
디코딩: '<|im_start|>assistant\n'


=== 5. DataCollator 설정 확인 ===
Collator response_template: '<|im_start|>as

In [63]:
from transformers import AutoTokenizer
from trl import DataCollatorForCompletionOnlyLM
import torch

tokenizer = AutoTokenizer.from_pretrained(model_name)

# === 테스트 1: response_template만 ===
print("="*80)
print("테스트 1: response_template만 사용")
print("="*80)

collator1 = DataCollatorForCompletionOnlyLM(
    response_template="<|im_start|>assistant\n",
    tokenizer=tokenizer,
    mlm=False
)

test_messages = dataset["train"][0]["messages"]
formatted_text = tokenizer.apply_chat_template(
    test_messages,
    tokenize=False,
    add_generation_prompt=False
)

tokenized = tokenizer(formatted_text, return_tensors="pt", truncation=True, max_length=512)
features1 = [{"input_ids": tokenized["input_ids"][0]}]
batch1 = collator1(features1)

labels1 = batch1['labels'][0]
learning1 = (labels1 != -100).sum().item()
print(f"학습 토큰: {learning1}")

# 학습되는 부분 확인
learning_mask1 = labels1 != -100
if learning_mask1.any():
    learning_ids1 = batch1['input_ids'][0][learning_mask1]
    print(f"학습 텍스트:\n{tokenizer.decode(learning_ids1, skip_special_tokens=False)[:200]}")

# === 테스트 2: instruction_template + response_template ===
print("\n" + "="*80)
print("테스트 2: instruction_template + response_template 사용")
print("="*80)

collator2 = DataCollatorForCompletionOnlyLM(
    instruction_template="<|im_start|>user\n",
    response_template="<|im_start|>assistant\n",
    tokenizer=tokenizer,
    mlm=False
)

features2 = [{"input_ids": tokenized["input_ids"][0]}]
batch2 = collator2(features2)

labels2 = batch2['labels'][0]
learning2 = (labels2 != -100).sum().item()
print(f"학습 토큰: {learning2}")

# 학습되는 부분 확인
learning_mask2 = labels2 != -100
if learning_mask2.any():
    learning_ids2 = batch2['input_ids'][0][learning_mask2]
    print(f"학습 텍스트:\n{tokenizer.decode(learning_ids2, skip_special_tokens=False)[:200]}")

# === 비교 ===
print("\n" + "="*80)
print("비교 결과")
print("="*80)
print(f"response_template만: {learning1} 토큰")
print(f"instruction + response: {learning2} 토큰")
print(f"차이: {learning2 - learning1} 토큰")


테스트 1: response_template만 사용
학습 토큰: 32
학습 텍스트:
레알 고마워 칭구, 이번 영화제 가서 가을낭만 제대로 느끼고 와야지 ㅋㅋ<|im_end|>


테스트 2: instruction_template + response_template 사용
학습 토큰: 173
학습 텍스트:
짜짜, 나도 가고 싶었는데 어땠어?<|im_end|>
어제가 개막식 이었어? 아까비 나도 개막식 가고 싶었는데, 이번 영화제에는 어떤 연예인이 참석해?<|im_end|>
대박, 나 이준기 엄청 좋아하잖아.<|im_end|>
나도 이번엔 무적권 가볼 거야, 언제가 폐막식이지?<|im_end|>
영화는 어디에서 볼 수 있는 거야?<|im_end|>
이번 영화제

비교 결과
response_template만: 32 토큰
instruction + response: 173 토큰
차이: 141 토큰


In [65]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

# 테스트 메시지
messages = [
    {'role': 'user', 'content': '안녕?'},
    {'role': 'assistant', 'content': '응ㅋㅋ'},
    {'role': 'user', 'content': '뭐해?'},
    {'role': 'assistant', 'content': '놀아'},
]

formatted = tokenizer.apply_chat_template(messages, tokenize=False)
tokenized = tokenizer(formatted, return_tensors="pt").to(model.device)

# Collator로 labels 생성
collator = DataCollatorForCompletionOnlyLM(
    instruction_template="<|im_start|>user\n",
    response_template="<|im_start|>assistant\n",
    tokenizer=tokenizer,
    mlm=False
)

batch = collator([{"input_ids": tokenized["input_ids"][0].tolist()}])
labels = batch['labels'].to(model.device)

print("=== Forward Pass 1회 ===\n")

# Forward pass (단 1번!)
with torch.no_grad():
    outputs = model(
        input_ids=tokenized['input_ids'],
        labels=labels,
        output_hidden_states=True,
    )

print(f"총 Loss: {outputs.loss.item():.4f}")
print(f"Forward pass 횟수: 1번")
print(f"학습된 토큰 수: {(labels[0] != -100).sum().item()}개")

# === Loss 상세 분석 ===
print("\n=== 위치별 Loss 계산 ===\n")

# 각 위치에서의 예측과 실제 타겟
input_ids = tokenized['input_ids'][0]
labels_single = labels[0]

# Logits: [seq_len, vocab_size]
logits = outputs.logits[0]

for i in range(len(input_ids) - 1):
    next_token_logit = logits[i]  # i번째 위치에서 i+1번째 예측
    next_token_pred = next_token_logit.argmax().item()
    next_token_true = input_ids[i + 1].item()
    next_token_label = labels_single[i + 1].item()
    
    # Label이 -100이 아닌 경우만 (학습 대상)
    if next_token_label != -100:
        pred_text = tokenizer.decode([next_token_pred])
        true_text = tokenizer.decode([next_token_true])
        
        # 이 위치의 loss 계산
        ce_loss = torch.nn.functional.cross_entropy(
            next_token_logit.unsqueeze(0),
            torch.tensor([next_token_true], device=model.device)
        )
        
        match = "✅" if next_token_pred == next_token_true else "❌"
        print(f"위치 {i:3d} → {i+1:3d}: "
              f"예측={repr(pred_text):10s} | "
              f"정답={repr(true_text):10s} | "
              f"Loss={ce_loss.item():.4f} {match}")

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the disk.


=== Forward Pass 1회 ===

총 Loss: 9.4761
Forward pass 횟수: 1번
학습된 토큰 수: 9개

=== 위치별 Loss 계산 ===

위치  10 →  11: 예측='안'        | 정답='응'        | Loss=17.7500 ❌
위치  11 →  12: 예측=','        | 정답='ㅋ'        | Loss=14.6250 ❌
위치  12 →  13: 예측=' 안'       | 정답='ㅋ'        | Loss=6.0000 ❌
위치  13 →  14: 예측=' 안'       | 정답='<|im_end|>' | Loss=19.8750 ❌
위치  14 →  15: 예측='\n'       | 정답='\n'       | Loss=-0.0000 ✅
위치  26 →  27: 예측='지'        | 정답='놀'        | Loss=9.9375 ❌
위치  27 →  28: 예측='이'        | 정답='아'        | Loss=2.0312 ❌
위치  28 →  29: 예측='요'        | 정답='<|im_end|>' | Loss=15.0625 ❌
위치  29 →  30: 예측='\n'       | 정답='\n'       | Loss=-0.0000 ✅


### Test

In [19]:
import sys
import os

project_root = "/Users/jangsehyeon/Desktop/Boostcamp/FINAL_PROJECT/pro-nlp-finalproject-nlp-13"
sys.path.append(project_root)

In [21]:
from src.sft.data.data_loader import load_raw_dataset
from src.sft.data.preprocessor import parse_and_add_system_prompt
from src.sft.data.collator import get_collator_and_format_func

In [27]:
data_path = "/Users/jangsehyeon/Downloads/sft_dataset.csv"
test_size = 0.1

raw_dataset = load_raw_dataset(data_path, test_size=test_size)

print("데이터 로드 및 분할 성공")
print(raw_dataset)
print(f"Train Sample: {len(raw_dataset['train'])}")
print(f"Test Sample: {len(raw_dataset['test'])}")

# 데이터 내용 살짝 보기 (아직 문자열 상태일 수 있음)
print("\n[Raw Data Sample]")
print(raw_dataset['train'][0]['messages'])

Loading raw data from /Users/jangsehyeon/Downloads/sft_dataset.csv...
데이터 로드 및 분할 성공
DatasetDict({
    train: Dataset({
        features: ['conversation_id', 'messages'],
        num_rows: 34500
    })
    test: Dataset({
        features: ['conversation_id', 'messages'],
        num_rows: 3834
    })
})
Train Sample: 34500
Test Sample: 3834

[Raw Data Sample]
[{'role': 'user', 'content': '어제 남친이랑 제10회 춘천영화제 다녀왔잖아.'}, {'role': 'assistant', 'content': '짜짜, 나도 가고 싶었는데 어땠어?'}, {'role': 'user', 'content': '어제 개막식이라 사람들 엄청 많드라.'}, {'role': 'assistant', 'content': '어제가 개막식 이었어? 아까비 나도 개막식 가고 싶었는데, 이번 영화제에는 어떤 연예인이 참석해?'}, {'role': 'user', 'content': '옥자연이 사회를 맡고 이준익 감독, 안성기, 박중훈, 이준기 배우가 관객과의 대화에 참여 한대.'}, {'role': 'assistant', 'content': '대박, 나 이준기 엄청 좋아하잖아.'}, {'role': 'user', 'content': 'ㅋㅋ 너 생각 나더라, 이준기 연기 정말 잘하지.'}, {'role': 'assistant', 'content': '나도 이번엔 무적권 가볼 거야, 언제가 폐막식이지?'}, {'role': 'user', 'content': '9월 11일 월요일까지 아직 시간 있으니까 꼭 가봐.'}, {'role': 'assistant', 'content': '영화는 어디에서 볼 

In [28]:
system_prompt = "너는 사용자의 친구처럼 편하게 대화하는 AI야. 말투는 자연스럽고 캐주얼하게, 너무 과장하지는 마. 필요할 땐 핵심을 잘 설명해주고, 사용자가 진지하면 너도 진지하게 답해. ㅋㅋ/이모지는 가끔만 써."

processed_dataset = parse_and_add_system_prompt(raw_dataset, system_prompt)

print("전처리 완료")


sample_messages = processed_dataset['train'][0]['messages']
print("\n[Processed Data Sample]")
for msg in sample_messages:
    print(f"- {msg['role']}: {msg['content'][:50]}...")

assert isinstance(sample_messages, list), "메시지가 리스트가 아닙니다!"
assert sample_messages[0]['role'] == 'system', "시스템 프롬프트가 맨 앞에 없습니다!"
print("\n🎉 데이터 구조 검증 통과!")

[Applying preprocessing]: Parsing & Adding System Prompt ('너는 사용자의 친구처럼 편하게 대화하는 AI야. 말투는 자연스럽고 캐주얼하게, 너무 과장하지는 마. 필요할 땐 핵심을 잘 설명해주고, 사용자가 진지하면 너도 진지하게 답해. ㅋㅋ/이모지는 가끔만 써....')...


Map:   0%|          | 0/34500 [00:00<?, ? examples/s]

Map:   0%|          | 0/3834 [00:00<?, ? examples/s]

전처리 완료

[Processed Data Sample]
- system: 너는 사용자의 친구처럼 편하게 대화하는 AI야. 말투는 자연스럽고 캐주얼하게, 너무 과장하...
- user: 어제 남친이랑 제10회 춘천영화제 다녀왔잖아....
- assistant: 짜짜, 나도 가고 싶었는데 어땠어?...
- user: 어제 개막식이라 사람들 엄청 많드라....
- assistant: 어제가 개막식 이었어? 아까비 나도 개막식 가고 싶었는데, 이번 영화제에는 어떤 연예인이 ...
- user: 옥자연이 사회를 맡고 이준익 감독, 안성기, 박중훈, 이준기 배우가 관객과의 대화에 참여 ...
- assistant: 대박, 나 이준기 엄청 좋아하잖아....
- user: ㅋㅋ 너 생각 나더라, 이준기 연기 정말 잘하지....
- assistant: 나도 이번엔 무적권 가볼 거야, 언제가 폐막식이지?...
- user: 9월 11일 월요일까지 아직 시간 있으니까 꼭 가봐....
- assistant: 영화는 어디에서 볼 수 있는 거야?...
- user: 공지천 아울러 까페와 메가박스 남춘천점, 커먼즈필드 안녕하우스에서 상영돼....
- assistant: 이번 영화제 주제는 뭐야?...
- user: 영화의 봄, 낭만의 가을이 이번 영화제 주제야....
- assistant: 내가 좋아하는 로맨스 드라마 많았으면 좋겠다....
- user: 응, 요일별 상영 영화 보내줄 테니까 보고 가봐....
- assistant: 레알 고마워 칭구, 이번 영화제 가서 가을낭만 제대로 느끼고 와야지 ㅋㅋ...

🎉 데이터 구조 검증 통과!


In [31]:
from transformers import AutoTokenizer

model_name = "Qwen/Qwen3-4B-Instruct-2507"
print(f"토크나이저 로딩 중... ({model_name})")

tokenizer = AutoTokenizer.from_pretrained(model_name)
# 패딩 토큰 설정 (보통 eos_token 사용)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Collator 및 Format 함수 가져오기
collator, format_func = get_collator_and_format_func(tokenizer)

print("토크나이저 및 Collator 준비 완료")

토크나이저 로딩 중... (Qwen/Qwen3-4B-Instruct-2507)
토크나이저 및 Collator 준비 완료


In [32]:
# 포맷팅 함수 실행
formatted_samples = format_func(processed_dataset['train'][:2])

print("\n=== 📝 최종 학습 데이터 포맷 예시 (Top 2) ===")
for i, text in enumerate(formatted_samples):
    print(f"\n[Sample {i+1}]")
    print(text)
    print("-" * 50)

# 체크 포인트
if "<|im_start|>system" in formatted_samples[0] and "<|im_start|>assistant" in formatted_samples[0]:
    print("\n✅ ChatML 포맷이 정상적으로 적용되었습니다.")
else:
    print("\n⚠️ 경고: ChatML 포맷이 보이지 않습니다. 템플릿을 확인하세요.")


=== 📝 최종 학습 데이터 포맷 예시 (Top 2) ===

[Sample 1]
<|im_start|>system
너는 사용자의 친구처럼 편하게 대화하는 AI야. 말투는 자연스럽고 캐주얼하게, 너무 과장하지는 마. 필요할 땐 핵심을 잘 설명해주고, 사용자가 진지하면 너도 진지하게 답해. ㅋㅋ/이모지는 가끔만 써.<|im_end|>
<|im_start|>user
어제 남친이랑 제10회 춘천영화제 다녀왔잖아.<|im_end|>
<|im_start|>assistant
짜짜, 나도 가고 싶었는데 어땠어?<|im_end|>
<|im_start|>user
어제 개막식이라 사람들 엄청 많드라.<|im_end|>
<|im_start|>assistant
어제가 개막식 이었어? 아까비 나도 개막식 가고 싶었는데, 이번 영화제에는 어떤 연예인이 참석해?<|im_end|>
<|im_start|>user
옥자연이 사회를 맡고 이준익 감독, 안성기, 박중훈, 이준기 배우가 관객과의 대화에 참여 한대.<|im_end|>
<|im_start|>assistant
대박, 나 이준기 엄청 좋아하잖아.<|im_end|>
<|im_start|>user
ㅋㅋ 너 생각 나더라, 이준기 연기 정말 잘하지.<|im_end|>
<|im_start|>assistant
나도 이번엔 무적권 가볼 거야, 언제가 폐막식이지?<|im_end|>
<|im_start|>user
9월 11일 월요일까지 아직 시간 있으니까 꼭 가봐.<|im_end|>
<|im_start|>assistant
영화는 어디에서 볼 수 있는 거야?<|im_end|>
<|im_start|>user
공지천 아울러 까페와 메가박스 남춘천점, 커먼즈필드 안녕하우스에서 상영돼.<|im_end|>
<|im_start|>assistant
이번 영화제 주제는 뭐야?<|im_end|>
<|im_start|>user
영화의 봄, 낭만의 가을이 이번 영화제 주제야.<|im_end|>
<|im_start|>assistant
내가 좋아하는 로맨스 드라마 많았으면 좋

In [33]:
import torch

# 1. 텍스트를 토큰화 (Trainer 내부 동작 시뮬레이션)
sample_text = formatted_samples[0]
tokenized_input = tokenizer(
    sample_text, 
    return_tensors="pt", 
    padding=True, 
    add_special_tokens=False
)

# 2. Collator에 넣기 위해 리스트 형태로 변환
# DataCollatorForCompletionOnlyLM은 'input_ids' 필드를 기대합니다.
dummy_batch = [{"input_ids": tokenized_input["input_ids"][0]}]

# 3. Collator 실행
batch_output = collator(dummy_batch)

# 4. Labels 확인 (Masking Check)
labels = batch_output["labels"][0]
input_ids = batch_output["input_ids"][0]

print("\n=== 🎭 마스킹 테스트 (User 부분은 -100이어야 함) ===")
print(f"Total Tokens: {len(labels)}")

# -100인 개수 (학습 안 하는 토큰)
ignored_tokens = (labels == -100).sum().item()
trained_tokens = (labels != -100).sum().item()

print(f"Ignored Tokens (User/System): {ignored_tokens}")
print(f"Trained Tokens (Assistant): {trained_tokens}")

if trained_tokens > 0:
    print("✅ Assistant 답변 부분이 정상적으로 레이블링 되었습니다.")
else:
    print("❌ 학습할 토큰이 없습니다! 템플릿 매칭에 실패했을 수 있습니다.")


=== 🎭 마스킹 테스트 (User 부분은 -100이어야 함) ===
Total Tokens: 510
Ignored Tokens (User/System): 329
Trained Tokens (Assistant): 181
✅ Assistant 답변 부분이 정상적으로 레이블링 되었습니다.


In [37]:
from src.sft.training.model_loader import load_model_and_tokenizer,ModelConfig,TrainConfig

In [ ]:
model, tokenizer = load_model_and_tokenizer(model_cfg, train_cfg)

In [ ]:
# 1. 양자화 확인 (Linear4bit 레이어가 보여야 함)
print("=== 🔍 모델 구조 확인 (일부) ===")
# 보통 첫 번째 레이어의 attention 쪽을 봅니다.
try:
    # Llama-3 기준 레이어 접근 경로
    sample_layer = model.model.layers[0].self_attn.q_proj
    print(f"Layer Type: {type(sample_layer)}")
    
    from bitsandbytes.nn import Linear4bit
    if isinstance(sample_layer, Linear4bit):
        print("✅ QLoRA(4bit) 적용됨: Linear4bit 레이어입니다.")
    else:
        print(f"⚠️ 경고: 4bit 레이어가 아닙니다. ({type(sample_layer)})")
except AttributeError:
    print("모델 구조가 달라서 직접 확인이 어렵습니다. print(model)을 해보세요.")

# 2. 데이터 타입 확인 (V100이니 float16이어야 함)
print(f"\n=== 💾 데이터 타입 확인 (V100 Check) ===")
print(f"Model dtype: {model.dtype}")
# compute_dtype은 레이어 내부에 숨어있어서 직접 확인은 어렵지만,
# model.dtype이 float16이면 일단 성공입니다.

# 3. 토크나이저 패딩 확인
print(f"\n=== 🧩 토크나이저 확인 ===")
print(f"Pad Token: {tokenizer.pad_token} (ID: {tokenizer.pad_token_id})")
print(f"EOS Token: {tokenizer.eos_token} (ID: {tokenizer.eos_token_id})")

if tokenizer.pad_token_id == tokenizer.eos_token_id:
    print("✅ Pad Token이 EOS Token으로 잘 설정되었습니다.")
else:
    print("⚠️ 주의: Pad != EOS (의도한 게 아니라면 확인 필요)")

In [ ]:
import sys
import os
import shutil
from unittest.mock import MagicMock
from datasets import Dataset

# 1. 경로 설정 (프로젝트 루트 찾기)
current_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(current_dir, "..")) if "src" not in os.listdir(current_dir) else current_dir
if project_root not in sys.path:
    sys.path.append(project_root)

# 2. 모듈 임포트
try:
    from src.sft.training.trainer import (
        build_sft_trainer, 
        SftTrainConfig, 
        CustomLoraConfig
    )
    from trl import SFTTrainer
    print("✅ 모듈 임포트 성공!")
except ImportError as e:
    print(f"❌ 임포트 실패: {e}")
    print("trainer.py 파일의 위치와 내용을 확인해주세요.")

# 3. 더미(Dummy) 객체 생성
# 실제 모델/토크나이저를 로드하면 오래 걸리므로, 테스트를 위해 껍데기만 만듭니다.
print("\n🛠️ 더미 데이터 및 객체 생성 중...")

# (1) Mock Tokenizer
mock_tokenizer = MagicMock()
mock_tokenizer.pad_token = "<pad>"
mock_tokenizer.model_max_length = 1024
# apply_chat_template이 호출되면 그냥 더미 텍스트 반환하도록 설정
mock_tokenizer.apply_chat_template.return_value = "User: Hello\nAssistant: Hi"

# (2) Mock Model
# SFTTrainer가 모델 타입을 체크하므로 PreTrainedModel을 상속받은 척하는 Mock 생성
from transformers import PreTrainedModel, PretrainedConfig
class MockModel(PreTrainedModel):
    def __init__(self):
        config = PretrainedConfig()
        super().__init__(config)
    def forward(self, *args, **kwargs):
        return {} # 더미 출력

mock_model = MockModel()

# (3) Dummy Dataset
dummy_data = [
    {"messages": [{"role": "user", "content": "hi"}, {"role": "assistant", "content": "hello"}]},
    {"messages": [{"role": "user", "content": "bye"}, {"role": "assistant", "content": "goodbye"}]}
]
train_dataset = Dataset.from_list(dummy_data)
eval_dataset = Dataset.from_list(dummy_data)

# (4) Config 설정 (V100 테스트)
sft_config = SftTrainConfig(
    output_dir="./test_output_trash", # 테스트 후 삭제할 경로
    num_train_epochs=1,
    fp16=True,   # V100 체크 포인트
    bf16=False,
    packing=False
)

lora_config = CustomLoraConfig(
    enabled=True,
    r=8,
    target_modules=["q_proj", "v_proj"] # 더미 타겟
)

# 4. Trainer 빌드 실행
print("\n🚀 build_sft_trainer 실행...")
try:
    trainer = build_sft_trainer(
        model=mock_model,
        tokenizer=mock_tokenizer,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        sft_cfg=sft_config,
        lora_cfg=lora_config,
        collator=None, # 테스트에선 생략 가능
        formatting_func=None # 내부 혹은 외부 주입 (여기선 None이어도 trainer 빌드 자체는 성공해야 함)
    )
    print("✅